# Auditory Supersitious Perception Writeup

**Focus questions**:

1. Full sentence vs imagined sentence (the two block types).
2. Oberlin vs UChicago site comparison.
3. Block order (least important).

Every analysis is reported THREE times: UChicago alone, Oberlin alone,
Unified.  Where line-fits are involved (questionnaire scatters), each
of the three groups gets its OWN scatter and its OWN regression line
in its OWN panel -- never one combined line over coloured dots.

## 1.  Imports, constants, and output directories

In [1]:
# ============================================================================
# Imports
# ============================================================================

from pathlib import Path

import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt

from scipy.stats import (wilcoxon, mannwhitneyu, spearmanr,
                          ttest_1samp, ttest_ind, ttest_rel)


# ============================================================================
# Project paths
# ============================================================================

CURRENT_DIRECTORY = Path(__file__).parent.resolve()
FIGURES_PATH      = CURRENT_DIRECTORY / "figures"
TABLES_PATH       = CURRENT_DIRECTORY / "tables"
ANALYSES_OUT_PATH = CURRENT_DIRECTORY / "stimuli-analyses" / "outputs"
RAW_DATA_PATH     = CURRENT_DIRECTORY / "raw_data"

FIGURES_PATH.mkdir(parents = True, exist_ok = True)
TABLES_PATH.mkdir(parents = True, exist_ok = True)
ANALYSES_OUT_PATH.mkdir(parents = True, exist_ok = True)


# ============================================================================
# Constants used throughout the notebook.
# ============================================================================

QUESTIONNAIRE_COLUMNS = ["BAIS_V", "BAIS_C", "VHQ", "TAS", "LSHS", "DES",
                         "FSS", "SSS_mean"]

SITE_COLOR  = {"UChicago": "#9467bd",
               "Oberlin":  "#2ca02c",
               "Unified":  "#1f77b4"}

BLOCK_LABEL = {"full_sentence":     "Full Sentence",
               "imagined_sentence": "Imagined Sentence"}

# Single seeded RNG used for every bootstrap / permutation / sub-sample
# in this notebook so that re-running it produces identical CSVs.
RNG = np.random.default_rng(20260501)


print(f"PROJECT_ROOT      = {PROJECT_ROOT}")
print(f"FIGURES_PATH      = {FIGURES_PATH}")
print(f"TABLES_PATH       = {TABLES_PATH}")
print(f"ANALYSES_OUT_PATH = {ANALYSES_OUT_PATH}")
print(f"RAW_DATA_PATH     = {RAW_DATA_PATH}")


NameError: name '__file__' is not defined

## 2.  Inlined data loaders

`run_analyses.py` imports `load_questionnaires` and `load_block_order`
from `stimuli-analyses/book3_helpers.py`.  To keep this notebook fully
self-contained, both functions are reproduced verbatim below (with
variable names spelled out) -- no external project imports.

In [ ]:
# ============================================================================
# Block-order loader
#   Reads the per-subject `summary_data_<NNN>.csv` files and emits one
#   row per (subject, block_type) with whether that block was run first
#   or second.
# ============================================================================

def _read_summary_csv(path):
    """One summary CSV -> dict of subject + which scheme was block 1/2."""
    df  = pd.read_csv(path)
    row = df.iloc[0]
    return dict(subject        = str(int(row["Subject Number"])),
                block_1_scheme = row["Block 1"],
                block_1_dprime = float(row["Block 1 D-Prime"]),
                block_2_scheme = row["Block 2"],
                block_2_dprime = float(row["Block 2 D-Prime"]))


def load_block_order():
    """Return long-form DataFrame:
        site, subject, block_type, block_order_index (1 or 2),
        summary_dprime."""
    rows = []
    for site in ("Oberlin", "UChicago"):
        site_root = RAW_DATA_PATH / site
        for subject_dir in sorted(p for p in site_root.iterdir() if p.is_dir()):
            subject_padded = subject_dir.name
            summary_files  = list(
                subject_dir.glob(f"summary_data_{subject_padded}.csv"))
            if not summary_files:
                continue
            summary = _read_summary_csv(summary_files[0])
            rows.append(dict(site              = site,
                             subject           = summary["subject"],
                             block_type        = summary["block_1_scheme"],
                             block_order_index = 1,
                             summary_dprime    = summary["block_1_dprime"]))
            rows.append(dict(site              = site,
                             subject           = summary["subject"],
                             block_type        = summary["block_2_scheme"],
                             block_order_index = 2,
                             summary_dprime    = summary["block_2_dprime"]))
    return pd.DataFrame(rows)


In [ ]:
# ============================================================================
# Questionnaire loader
#   For each per-subject directory, sum each questionnaire's Q1..Qn
#   responses and compute a mean.  Stanford Sleepiness Scale is handled
#   separately (per-block pre/post timepoints).
# ============================================================================

# (file_stem, n_items, score_name, reverse_indices)
QUESTIONNAIRE_DEFINITIONS = [
    ("bais_v",                   14, "BAIS_V",  []),  # auditory imagery vividness
    ("bais_c",                   14, "BAIS_C",  []),  # auditory imagery control
    ("vhq",                      14, "VHQ",     []),
    ("tellegen",                 34, "TAS",     []),  # absorption
    ("launay_slade",             16, "LSHS",    []),  # hallucination prone
    ("dissociative_experiences", 28, "DES",     []),
    ("flow_state_scale",          9, "FSS",     []),
]


def _read_one_questionnaire(path, n_items):
    """Return total / mean / n_valid for the first row of a questionnaire CSV."""
    df    = pd.read_csv(path)
    row   = df.iloc[0]
    cols  = [f"Q{i}" for i in range(1, n_items + 1)]
    vals  = pd.to_numeric(row[cols], errors = "coerce")
    return dict(total   = float(vals.sum()),
                mean    = float(vals.mean()),
                n_valid = int(vals.notna().sum()))


def load_questionnaires():
    """Wide DataFrame: site, subject, plus one column per questionnaire
    (TOTAL score) plus per-questionnaire `*_mean` columns and
    SSS_mean / SSS_pre_b1 / SSS_post_b2 for the Stanford Sleepiness Scale."""
    rows = []
    for site in ("Oberlin", "UChicago"):
        site_root = RAW_DATA_PATH / site
        for subject_dir in sorted(p for p in site_root.iterdir() if p.is_dir()):
            subject_padded = subject_dir.name
            entry = dict(site = site, subject = str(int(subject_padded)))

            # Sum-score questionnaires
            for stem, n_items, score_name, _ in QUESTIONNAIRE_DEFINITIONS:
                quest_path = subject_dir / f"{stem}_{subject_padded}.csv"
                if not quest_path.exists():
                    entry[score_name] = np.nan
                    continue
                summary = _read_one_questionnaire(quest_path, n_items)
                entry[score_name]            = summary["total"]
                entry[f"{score_name}_mean"]  = summary["mean"]

            # Stanford Sleepiness Scale -- per-block pre/post timepoints
            sss_path = subject_dir / f"stanford_sleepiness_{subject_padded}.csv"
            if sss_path.exists():
                sss = pd.read_csv(sss_path)
                sss["response"] = pd.to_numeric(sss["response"],
                                                 errors = "coerce")
                entry["SSS_mean"]    = float(sss["response"].mean())
                entry["SSS_pre_b1"]  = float(
                    sss[(sss["block_index"] == 1)
                         & (sss["pre_or_post"] == "pre")]["response"].mean())
                entry["SSS_post_b2"] = float(
                    sss[(sss["block_index"] == 2)
                         & (sss["pre_or_post"] == "post")]["response"].mean())
            rows.append(entry)
    return pd.DataFrame(rows)


## 3.  Build the master per-subject table

One row per (subject, block_type), with site, accuracy, $d'$, the two
cosine alignment metrics, the per-subject logistic
$\beta_{\text{true target}}$, the eight questionnaires, and the
block-order index.  Every later section pulls from this single
DataFrame.

In [ ]:
# ============================================================================
# Master loader: merges per-subject SDT, classification image, spectral,
# reminder-anchored slope, questionnaires, and block-order.
# ============================================================================

def load_full_per_subject_table():
    """One row per (subject, block_type) with everything we need."""
    sdt_table      = pd.read_csv(ANALYSES_OUT_PATH / "book2"
                                  / "ch16-sdt"
                                  / "ch16_per_subject_sdt.csv")
    ci_table       = pd.read_csv(ANALYSES_OUT_PATH / "book3"
                                  / "ch22-classification-image"
                                  / "ch22_per_subject_ci.csv")
    spectral_table = pd.read_csv(ANALYSES_OUT_PATH / "book3"
                                  / "ch24-spectral-logistic"
                                  / "ch24_per_subject_spectral.csv")
    slope_table    = pd.read_csv(ANALYSES_OUT_PATH / "book3"
                                  / "ch25-reminder-anchored"
                                  / "ch25_per_subject_slopes.csv")

    for table in (sdt_table, ci_table, spectral_table, slope_table):
        table["subject"] = table["subject"].astype(str)

    merged = (sdt_table[["site", "subject", "block_type",
                          "hit_rate", "fa_rate", "d_prime", "criterion",
                          "accuracy", "p_perm_dprime"]]
                .merge(ci_table[["subject", "block_type", "cos_ci_true",
                                  "p_perm_pos"]]
                        .rename(columns = {"p_perm_pos": "p_perm_ci"}),
                        on = ["subject", "block_type"])
                .merge(spectral_table[["subject", "block_type",
                                        "cos_w_true", "p_perm_pos"]]
                        .rename(columns = {"p_perm_pos": "p_perm_spec"}),
                        on = ["subject", "block_type"])
                .merge(slope_table[["subject", "block_type",
                                     "beta_truetarget"]],
                        on = ["subject", "block_type"]))

    questionnaires = load_questionnaires()
    questionnaires["subject"] = questionnaires["subject"].astype(str)
    merged = merged.merge(
        questionnaires[["subject"] + QUESTIONNAIRE_COLUMNS],
        on = "subject", how = "left")

    block_order = load_block_order()
    block_order["subject"] = block_order["subject"].astype(str)
    merged = merged.merge(
        block_order[["site", "subject", "block_type", "block_order_index"]],
        on = ["site", "subject", "block_type"], how = "left")

    return merged


per_subject_table = load_full_per_subject_table()
print(f"Loaded {len(per_subject_table)} rows "
      f"({per_subject_table['subject'].nunique()} subjects, "
      f"sites = {dict(per_subject_table.groupby('site')['subject'].nunique())})")
per_subject_table.head()


## 4.  Statistical helpers

All of the small calculations the report uses (Cohen's $d$, bootstrap
CIs on the mean, sign-flip / shuffle permutation tests, three-way
group splits) live here so the per-section blocks below stay readable.

In [ ]:
# ============================================================================
# Cohen's d helpers
#   - one_sample : deviation of the mean from a fixed null (in SDs of x)
#   - two_sample : pooled-SD form for unpaired groups
# ============================================================================

def cohens_d_one_sample(values, null_mean = 0.0):
    values = np.asarray(values, dtype = float)
    return (np.mean(values) - null_mean) / np.std(values, ddof = 1)


def cohens_d_two_sample(group_a, group_b):
    group_a = np.asarray(group_a, dtype = float)
    group_b = np.asarray(group_b, dtype = float)
    pooled_sd = np.sqrt(
        ((len(group_a) - 1) * np.var(group_a, ddof = 1)
         + (len(group_b) - 1) * np.var(group_b, ddof = 1))
        / (len(group_a) + len(group_b) - 2))
    return (np.mean(group_a) - np.mean(group_b)) / pooled_sd


In [ ]:
# ============================================================================
# Bootstrap percentile CI on the mean.
#   Default = 10000 resamples, 95% percentile interval.
#   Returns (mean, ci_lo, ci_hi).
# ============================================================================

def bootstrap_ci_of_mean(values, n_boot = 10000, ci_percent = 95, rng = RNG):
    values     = np.asarray(values, dtype = float)
    boot_means = rng.choice(values, size = (n_boot, len(values)),
                             replace = True).mean(axis = 1)
    lower_pct  = (100 - ci_percent) / 2
    upper_pct  = 100 - lower_pct
    return (float(np.mean(values)),
            float(np.percentile(boot_means, lower_pct)),
            float(np.percentile(boot_means, upper_pct)))


In [ ]:
# ============================================================================
# Permutation test on a mean difference.
#   - paired = True  : sign-flip the within-subject differences a-b
#   - paired = False : shuffle pooled (a, b) and re-split
#   Returns (observed_diff, two_sided_p).
# ============================================================================

def permutation_test_mean_difference(group_a, group_b,
                                      n_perm = 10000, rng = RNG,
                                      paired = False):
    group_a = np.asarray(group_a, dtype = float)
    group_b = np.asarray(group_b, dtype = float)
    if paired:
        differences   = group_a - group_b
        observed_diff = float(np.mean(differences))
        sign_flips    = rng.choice([-1, 1],
                                    size = (n_perm, len(differences)))
        permuted      = (sign_flips * differences).mean(axis = 1)
    else:
        observed_diff = float(np.mean(group_a) - np.mean(group_b))
        pooled        = np.concatenate([group_a, group_b])
        n_a           = len(group_a)
        permuted      = np.empty(n_perm)
        for i in range(n_perm):
            rng.shuffle(pooled)
            permuted[i] = pooled[:n_a].mean() - pooled[n_a:].mean()
    p_two_sided = float(
        (np.abs(permuted) >= np.abs(observed_diff) - 1e-12).mean())
    return observed_diff, p_two_sided


In [ ]:
# ============================================================================
# Three-way group splitter.
#   Given a per-subject DataFrame and a block name, return a dict with
#   keys "UChicago", "Oberlin", "Unified" mapping to the corresponding
#   per-subject sub-DataFrames.
# ============================================================================

def split_into_three_groups(table, block_type):
    block_subset = table[table["block_type"] == block_type]
    return {"UChicago": block_subset[block_subset["site"] == "UChicago"],
            "Oberlin":  block_subset[block_subset["site"] == "Oberlin"],
            "Unified":  block_subset}


## 5.  Section A -- accuracy vs chance, per group, per block

Base question: in each block, in each group, are subjects performing
reliably above chance ($\text{accuracy} = 0.5$)?  We report
one-sample Wilcoxon (primary, since $N$ is small), one-sample $t$-test
(parametric counterpart), bootstrap 95% CI on the mean, and Cohen's
$d$ relative to the chance value.

In [ ]:
# ============================================================================
# Section A -- table of accuracy vs chance (six rows: 2 blocks x 3 groups)
# ============================================================================

print("\n=== Section A: accuracy vs chance (per group, per block) ===")

accuracy_vs_chance_rows = []
for block_type in ("full_sentence", "imagined_sentence"):
    groups_by_site = split_into_three_groups(per_subject_table, block_type)
    for group_name, group_subset in groups_by_site.items():
        accuracies                   = group_subset["accuracy"].values
        mean_acc, ci_lower, ci_upper = bootstrap_ci_of_mean(accuracies)
        try:
            wilcoxon_stat, wilcoxon_p_value = wilcoxon(accuracies - 0.5)
        except ValueError:
            wilcoxon_stat, wilcoxon_p_value = (np.nan, np.nan)
        t_stat, t_p_value            = ttest_1samp(accuracies, 0.5)
        cohens_d                     = cohens_d_one_sample(accuracies, 0.5)
        accuracy_vs_chance_rows.append(dict(
            block      = block_type,
            group      = group_name,
            n          = len(accuracies),
            mean       = mean_acc,
            sd         = float(np.std(accuracies, ddof = 1)),
            ci_lo      = ci_lower,
            ci_hi      = ci_upper,
            wilcoxon_W = float(wilcoxon_stat),
            wilcoxon_p = float(wilcoxon_p_value),
            t          = float(t_stat),
            t_p        = float(t_p_value),
            cohens_d   = float(cohens_d)))

accuracy_vs_chance_table = pd.DataFrame(accuracy_vs_chance_rows)
accuracy_vs_chance_table.to_csv(TABLES_PATH / "A_accuracy_vs_chance.csv",
                                 index = False)
print(accuracy_vs_chance_table.round(4).to_string(index = False))


In [ ]:
# ============================================================================
# Section A -- accuracy distribution figure (2 blocks x 3 groups)
#   Each panel:  histogram of per-subject accuracies
#                red dashed line  = chance (0.50)
#                black solid line = group mean
# ============================================================================

fig, axes_grid = plt.subplots(2, 3, figsize = (12, 6.5), sharey = True)
for row_index, block_type in enumerate(("full_sentence", "imagined_sentence")):
    groups_by_site = split_into_three_groups(per_subject_table, block_type)
    for col_index, group_name in enumerate(("UChicago", "Oberlin", "Unified")):
        ax         = axes_grid[row_index, col_index]
        accuracies = groups_by_site[group_name]["accuracy"].values
        ax.hist(accuracies, bins = 12, color = SITE_COLOR[group_name],
                 alpha = 0.75, edgecolor = "k")
        ax.axvline(0.5, color = "red", linestyle = "--", linewidth = 1.5)
        ax.axvline(np.mean(accuracies), color = "black", linestyle = "-",
                    linewidth = 2)
        mean_acc, ci_lower, ci_upper = bootstrap_ci_of_mean(accuracies)
        row = accuracy_vs_chance_table.query(
            "block == @block_type and group == @group_name").iloc[0]
        ax.set_title(f"{group_name} -- {BLOCK_LABEL[block_type]}\n"
                      f"mean = {mean_acc:.3f}  "
                      f"[{ci_lower:.3f}, {ci_upper:.3f}]   "
                      f"Wilcoxon p = {row['wilcoxon_p']:.3f}",
                      fontsize = 9)
        ax.set_xlabel("accuracy")
        if col_index == 0:
            ax.set_ylabel("# subjects")
        ax.set_xlim(0.40, 0.60)
plt.suptitle("Section A.  Accuracy vs chance, three-way split per block",
              fontsize = 12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "A_accuracy_vs_chance.png", dpi = 140)
plt.show()


## 6.  Section B -- full sentence vs imagined sentence (paired)

Each subject ran both blocks, so the right test is paired.  We report
paired Wilcoxon (primary), paired $t$ (parametric counterpart), a
sign-flip permutation $p$-value, and Cohen's $d$ on the within-subject
differences.

In [ ]:
# ============================================================================
# Section B -- table of paired Full vs Imagined (one row per group)
# ============================================================================

print("\n=== Section B: full vs imagined (paired, 3-way) ===")

full_vs_imagined_rows = []
for group_name in ("UChicago", "Oberlin", "Unified"):
    if group_name == "Unified":
        group_subset = per_subject_table
    else:
        group_subset = per_subject_table[per_subject_table["site"]
                                          == group_name]
    wide_per_subject  = group_subset.pivot(index   = "subject",
                                            columns = "block_type",
                                            values  = "accuracy").dropna()
    accuracy_full     = wide_per_subject["full_sentence"].values
    accuracy_imagined = wide_per_subject["imagined_sentence"].values
    differences       = accuracy_full - accuracy_imagined

    wilcoxon_stat, wilcoxon_p_value = wilcoxon(differences)
    paired_t_stat, paired_t_p_value = ttest_rel(accuracy_full,
                                                 accuracy_imagined)
    observed_diff, perm_p_value     = permutation_test_mean_difference(
        accuracy_full, accuracy_imagined, paired = True)
    cohens_d_paired                 = cohens_d_one_sample(differences, 0.0)

    full_vs_imagined_rows.append(dict(
        group           = group_name,
        n               = len(differences),
        mean_full       = float(np.mean(accuracy_full)),
        mean_imag       = float(np.mean(accuracy_imagined)),
        mean_diff       = float(np.mean(differences)),
        sd_diff         = float(np.std(differences, ddof = 1)),
        wilcoxon_W      = float(wilcoxon_stat),
        wilcoxon_p      = float(wilcoxon_p_value),
        paired_t        = float(paired_t_stat),
        t_p             = float(paired_t_p_value),
        perm_p          = float(perm_p_value),
        cohens_d_paired = float(cohens_d_paired)))

full_vs_imagined_table = pd.DataFrame(full_vs_imagined_rows)
full_vs_imagined_table.to_csv(TABLES_PATH / "B_full_vs_imagined.csv",
                               index = False)
print(full_vs_imagined_table.round(4).to_string(index = False))


In [ ]:
# ============================================================================
# Section B -- per-subject within-subject lines, one panel per group
# ============================================================================

fig, axes_row = plt.subplots(1, 3, figsize = (12, 4.5))
for ax, group_name in zip(axes_row, ("UChicago", "Oberlin", "Unified")):
    if group_name == "Unified":
        group_subset = per_subject_table
    else:
        group_subset = per_subject_table[per_subject_table["site"]
                                          == group_name]
    wide_per_subject = group_subset.pivot(index   = "subject",
                                           columns = "block_type",
                                           values  = "accuracy").dropna()
    for _, subject_row in wide_per_subject.iterrows():
        ax.plot([0, 1],
                 [subject_row["full_sentence"],
                  subject_row["imagined_sentence"]],
                 color = "gray", alpha = 0.5, marker = "o")
    ax.plot([0, 1],
             [wide_per_subject["full_sentence"].mean(),
              wide_per_subject["imagined_sentence"].mean()],
             color = SITE_COLOR[group_name], linewidth = 3, marker = "s",
             markersize = 10, label = "group mean")
    ax.axhline(0.5, color = "red", linestyle = "--", alpha = 0.7)
    row = full_vs_imagined_table.query("group == @group_name").iloc[0]
    ax.set_title(f"{group_name}  (n = {int(row['n'])})\n"
                  f"diff = {row['mean_diff']:+.3f}, "
                  f"Wilcoxon p = {row['wilcoxon_p']:.3f}, "
                  f"d = {row['cohens_d_paired']:+.2f}", fontsize = 10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Full", "Imagined"])
    ax.set_ylabel("accuracy")
    ax.set_ylim(0.40, 0.60)
plt.suptitle("Section B.  Within-subject Full vs Imagined accuracy",
              fontsize = 12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "B_full_vs_imagined.png", dpi = 140)
plt.show()


## 7.  Section C -- Oberlin vs UChicago site comparison (unpaired)

One row per block.  Mann-Whitney $U$ is the primary test (small $N$,
non-Gaussian); Welch $t$ and a shuffle permutation $p$ are reported
for context, plus pooled-SD Cohen's $d$.

In [ ]:
# ============================================================================
# Section C -- table of unpaired site comparison per block
# ============================================================================

print("\n=== Section C: Oberlin vs UChicago per block ===")

site_comparison_rows = []
for block_type in ("full_sentence", "imagined_sentence"):
    block_subset      = per_subject_table[per_subject_table["block_type"]
                                           == block_type]
    accuracy_oberlin  = block_subset[block_subset["site"]
                                      == "Oberlin"]["accuracy"].values
    accuracy_uchicago = block_subset[block_subset["site"]
                                      == "UChicago"]["accuracy"].values

    mwu_stat,    mwu_p_value     = mannwhitneyu(accuracy_oberlin,
                                                 accuracy_uchicago,
                                                 alternative = "two-sided")
    welch_t_stat, welch_p_value  = ttest_ind(accuracy_oberlin,
                                              accuracy_uchicago,
                                              equal_var = False)
    observed_diff, perm_p_value  = permutation_test_mean_difference(
        accuracy_oberlin, accuracy_uchicago, paired = False)
    cohens_d                     = cohens_d_two_sample(accuracy_oberlin,
                                                        accuracy_uchicago)

    site_comparison_rows.append(dict(
        block         = block_type,
        n_oberlin     = len(accuracy_oberlin),
        n_uchicago    = len(accuracy_uchicago),
        mean_oberlin  = float(np.mean(accuracy_oberlin)),
        mean_uchicago = float(np.mean(accuracy_uchicago)),
        diff          = float(np.mean(accuracy_oberlin)
                                - np.mean(accuracy_uchicago)),
        mwu_U         = float(mwu_stat),
        mwu_p         = float(mwu_p_value),
        welch_t       = float(welch_t_stat),
        welch_p       = float(welch_p_value),
        perm_p        = float(perm_p_value),
        cohens_d      = float(cohens_d)))

site_comparison_table = pd.DataFrame(site_comparison_rows)
site_comparison_table.to_csv(TABLES_PATH / "C_site_comparison.csv",
                              index = False)
print(site_comparison_table.round(4).to_string(index = False))


In [ ]:
# ============================================================================
# Section C -- per-block site boxplots with overlaid jittered points
# ============================================================================

fig, axes_row = plt.subplots(1, 2, figsize = (10, 5))
for ax, block_type in zip(axes_row, ("full_sentence", "imagined_sentence")):
    block_subset = per_subject_table[per_subject_table["block_type"]
                                      == block_type]
    box_data, box_colors, box_labels = [], [], []
    for site_name in ("UChicago", "Oberlin"):
        accuracies = block_subset[block_subset["site"]
                                   == site_name]["accuracy"].values
        box_data  .append(accuracies)
        box_colors.append(SITE_COLOR[site_name])
        box_labels.append(f"{site_name}\n(n = {len(accuracies)})")
    bp = ax.boxplot(box_data, labels = box_labels, patch_artist = True,
                     widths = 0.55)
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color); patch.set_alpha(0.6)
    for box_index, accuracies in enumerate(box_data):
        ax.scatter(np.full_like(accuracies, box_index + 1, dtype = float)
                    + RNG.uniform(-0.06, 0.06, size = len(accuracies)),
                    accuracies, color = box_colors[box_index],
                    edgecolor = "k", zorder = 3)
    ax.axhline(0.5, color = "red", linestyle = "--", alpha = 0.7)
    row = site_comparison_table.query("block == @block_type").iloc[0]
    ax.set_title(f"{BLOCK_LABEL[block_type]}\n"
                  f"diff = {row['diff']:+.3f},  "
                  f"MW p = {row['mwu_p']:.3f},  "
                  f"d = {row['cohens_d']:+.2f}", fontsize = 10)
    ax.set_ylabel("accuracy")
    ax.set_ylim(0.40, 0.60)
plt.suptitle("Section C.  Site comparison per block (unpaired)",
              fontsize = 12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "C_site_comparison.png", dpi = 140)
plt.show()


## 8.  Section D -- block order

Subjects who ran a given block FIRST (block_order_index = 1) vs
subjects who ran it SECOND (block_order_index = 2).  Unpaired
Mann-Whitney inside each (block, group).  Cells with fewer than 3
subjects on either side are skipped.

In [ ]:
# ============================================================================
# Section D -- block order table
# ============================================================================

print("\n=== Section D: block order ===")

block_order_rows = []
for block_type in ("full_sentence", "imagined_sentence"):
    groups_by_site = split_into_three_groups(per_subject_table, block_type)
    for group_name, group_subset in groups_by_site.items():
        accuracy_first  = group_subset[group_subset["block_order_index"]
                                        == 1]["accuracy"].dropna().values
        accuracy_second = group_subset[group_subset["block_order_index"]
                                        == 2]["accuracy"].dropna().values
        if len(accuracy_first) < 3 or len(accuracy_second) < 3:
            block_order_rows.append(dict(
                block    = block_type,
                group    = group_name,
                n_b1     = len(accuracy_first),
                n_b2     = len(accuracy_second),
                mean_b1  = (float(np.mean(accuracy_first))
                             if len(accuracy_first) else np.nan),
                mean_b2  = (float(np.mean(accuracy_second))
                             if len(accuracy_second) else np.nan),
                mwu_p    = np.nan,
                cohens_d = np.nan))
            continue
        mwu_stat, mwu_p_value = mannwhitneyu(accuracy_first, accuracy_second,
                                              alternative = "two-sided")
        block_order_rows.append(dict(
            block    = block_type,
            group    = group_name,
            n_b1     = len(accuracy_first),
            n_b2     = len(accuracy_second),
            mean_b1  = float(np.mean(accuracy_first)),
            mean_b2  = float(np.mean(accuracy_second)),
            mwu_p    = float(mwu_p_value),
            cohens_d = float(cohens_d_two_sample(accuracy_second,
                                                  accuracy_first))))

block_order_table = pd.DataFrame(block_order_rows)
block_order_table.to_csv(TABLES_PATH / "D_block_order.csv", index = False)
print(block_order_table.round(4).to_string(index = False))


## 9.  Section E -- convergent metrics for the Oberlin full-sentence result

Five mathematically distinct per-subject quantities, all of which
should equal their null value if subjects are responding at random and
all of which should be positive if subjects are exploiting the
experimenter's acoustic structure.  Tested in the **full_sentence**
block, three-way split.

Metrics:
- $d'$ (signal-detection sensitivity), null = 0
- accuracy, null = 0.5
- $\cos(\mathrm{CI}_s, \mathrm{CI}_{\mathrm{true}})$, null = 0
- $\cos(w_s, w_{\mathrm{true}})$, null = 0  (32-band log-spectral)
- $\beta_{\text{true target}}$ from the per-subject reminder-anchored
  logistic, null = 0

In [ ]:
# ============================================================================
# Section E -- convergent metrics table
# ============================================================================

print("\n=== Section E: convergent metrics Oberlin full_sentence ===")

CONVERGENT_METRICS = [
    ("d_prime",         "d'  (SDT sensitivity, Ch.16)",                 0.0),
    ("accuracy",        "accuracy",                                      0.5),
    ("cos_ci_true",     "cos(CI_subject, CI_true) (Ch.22)",              0.0),
    ("cos_w_true",      "cos(w_subject, w_true) (Ch.24)",                0.0),
    ("beta_truetarget", "beta_truetarget (per-subject logistic, Ch.25)", 0.0),
]

convergent_rows = []
for group_name in ("UChicago", "Oberlin", "Unified"):
    group_subset = per_subject_table[per_subject_table["block_type"]
                                      == "full_sentence"]
    if group_name != "Unified":
        group_subset = group_subset[group_subset["site"] == group_name]
    for metric_column, metric_label, null_value in CONVERGENT_METRICS:
        metric_values = group_subset[metric_column].dropna().values
        try:
            wilcoxon_stat, wilcoxon_p_value = wilcoxon(metric_values
                                                        - null_value)
        except ValueError:
            wilcoxon_stat, wilcoxon_p_value = (np.nan, np.nan)
        t_stat, t_p_value              = ttest_1samp(metric_values, null_value)
        cohens_d                       = cohens_d_one_sample(metric_values,
                                                              null_value)
        mean_value, ci_lower, ci_upper = bootstrap_ci_of_mean(metric_values)
        convergent_rows.append(dict(
            group      = group_name,
            metric     = metric_column,
            label      = metric_label,
            n          = len(metric_values),
            null       = null_value,
            mean       = mean_value,
            ci_lo      = ci_lower,
            ci_hi      = ci_upper,
            wilcoxon_W = (float(wilcoxon_stat)
                          if not np.isnan(wilcoxon_stat) else np.nan),
            wilcoxon_p = (float(wilcoxon_p_value)
                          if not np.isnan(wilcoxon_p_value) else np.nan),
            t          = float(t_stat),
            t_p        = float(t_p_value),
            cohens_d   = float(cohens_d)))

convergent_metrics_table = pd.DataFrame(convergent_rows)
convergent_metrics_table.to_csv(TABLES_PATH / "E_convergent_full_sentence.csv",
                                 index = False)
print(convergent_metrics_table.round(4)[
    ["group", "metric", "n", "mean", "ci_lo", "ci_hi", "wilcoxon_p",
     "cohens_d"]].to_string(index = False))


In [ ]:
# ============================================================================
# Section E -- 5x3 grid of histograms
#   Rows    = metric
#   Columns = UChicago / Oberlin / Unified
#   Red dashed line  = null value (0 or 0.5)
#   Black solid line = group mean
# ============================================================================

fig, axes_grid = plt.subplots(len(CONVERGENT_METRICS), 3,
                               figsize = (12, 2.6 * len(CONVERGENT_METRICS)),
                               sharey = "row")
for row_index, (metric_column, metric_label, null_value) \
        in enumerate(CONVERGENT_METRICS):
    for col_index, group_name in enumerate(("UChicago", "Oberlin", "Unified")):
        ax            = axes_grid[row_index, col_index]
        group_subset  = per_subject_table[per_subject_table["block_type"]
                                           == "full_sentence"]
        if group_name != "Unified":
            group_subset = group_subset[group_subset["site"] == group_name]
        metric_values = group_subset[metric_column].dropna().values
        ax.hist(metric_values, bins = 10, color = SITE_COLOR[group_name],
                 alpha = 0.75, edgecolor = "k")
        ax.axvline(null_value, color = "red", linestyle = "--",
                    linewidth = 1.5)
        ax.axvline(np.mean(metric_values), color = "black", linestyle = "-",
                    linewidth = 2)
        row = convergent_metrics_table.query(
            "group == @group_name and metric == @metric_column").iloc[0]
        ax.set_title(f"{group_name}: {metric_label}\n"
                      f"mean = {row['mean']:+.3f}, "
                      f"Wilcoxon p = {row['wilcoxon_p']:.3f}, "
                      f"d = {row['cohens_d']:+.2f}", fontsize = 8)
        if row_index == len(CONVERGENT_METRICS) - 1:
            ax.set_xlabel("metric value")
plt.suptitle("Section E.  Four mathematically distinct alignment statistics, "
              "full_sentence block, three-way split", fontsize = 12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "E_convergent_full_sentence.png", dpi = 140)
plt.show()


## 10.  Section F -- questionnaire correlations (exploratory)

Spearman $\rho$ between each questionnaire and accuracy, in each
(block, group).  This is exploratory: $8 \times 2 \times 3 = 48$
correlations, Bonferroni at $\alpha = 0.05$ requires
$p < 1.04 \times 10^{-3}$.  Each questionnaire is also plotted as
scatter + per-panel OLS line; the report uses the top entries by
$p$-value.

In [ ]:
# ============================================================================
# Section F -- Spearman correlation table
# ============================================================================

print("\n=== Section F: questionnaire vs accuracy ===")

questionnaire_correlation_rows = []
for block_type in ("full_sentence", "imagined_sentence"):
    groups_by_site = split_into_three_groups(per_subject_table, block_type)
    for questionnaire_name in QUESTIONNAIRE_COLUMNS:
        for group_name, group_subset in groups_by_site.items():
            questionnaire_values = group_subset[questionnaire_name].values
            accuracy_values      = group_subset["accuracy"].values
            valid                = ~(np.isnan(questionnaire_values)
                                      | np.isnan(accuracy_values))
            if valid.sum() < 4:
                continue
            spearman_rho, spearman_p = spearmanr(
                questionnaire_values[valid], accuracy_values[valid])
            questionnaire_correlation_rows.append(dict(
                block = block_type,
                group = group_name,
                quest = questionnaire_name,
                n     = int(valid.sum()),
                rho   = float(spearman_rho),
                p     = float(spearman_p)))

questionnaire_correlation_table = pd.DataFrame(questionnaire_correlation_rows)
questionnaire_correlation_table.to_csv(
    TABLES_PATH / "F_questionnaire_spearman.csv", index = False)
print(questionnaire_correlation_table.sort_values("p").head(20)
        .round(4).to_string(index = False))


In [ ]:
# ============================================================================
# Section F -- one figure per block.
#   Grid: rows = questionnaires, cols = group (UChicago, Oberlin, Unified).
#   Each panel has its OWN scatter + its OWN OLS line.
# ============================================================================

for block_type in ("full_sentence", "imagined_sentence"):
    fig, axes_grid = plt.subplots(len(QUESTIONNAIRE_COLUMNS), 3,
                                   figsize = (11,
                                               2.4 * len(QUESTIONNAIRE_COLUMNS)))
    groups_by_site = split_into_three_groups(per_subject_table, block_type)
    for row_index, questionnaire_name in enumerate(QUESTIONNAIRE_COLUMNS):
        for col_index, group_name in enumerate(("UChicago", "Oberlin",
                                                  "Unified")):
            ax                   = axes_grid[row_index, col_index]
            group_subset         = groups_by_site[group_name]
            questionnaire_values = group_subset[questionnaire_name].values
            accuracy_values      = group_subset["accuracy"].values
            valid                = ~(np.isnan(questionnaire_values)
                                      | np.isnan(accuracy_values))
            xs = questionnaire_values[valid]
            ys = accuracy_values[valid]
            ax.scatter(xs, ys, color = SITE_COLOR[group_name],
                        edgecolor = "k", alpha = 0.85)
            if len(xs) >= 3 and np.ptp(xs) > 0:
                slope, intercept = np.polyfit(xs, ys, 1)
                line_x = np.array([xs.min(), xs.max()])
                ax.plot(line_x, slope * line_x + intercept,
                         color = SITE_COLOR[group_name], linestyle = "--",
                         linewidth = 2)
            spearman_rho, spearman_p = (spearmanr(xs, ys)
                                          if len(xs) >= 4
                                          else (np.nan, np.nan))
            ax.axhline(0.5, color = "red", linestyle = ":", alpha = 0.5)
            ax.set_title(f"{group_name}: {questionnaire_name}\n"
                          f"n = {len(xs)}, "
                          f"rho = {spearman_rho:+.2f}, "
                          f"p = {spearman_p:.3f}", fontsize = 8)
            if col_index == 0:
                ax.set_ylabel("accuracy")
            if row_index == len(QUESTIONNAIRE_COLUMNS) - 1:
                ax.set_xlabel(questionnaire_name)
    plt.suptitle(f"Section F.  Questionnaires vs accuracy, "
                  f"{BLOCK_LABEL[block_type]}.  "
                  "Each panel has its own group, scatter and OLS line.",
                  fontsize = 11)
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / f"F_quest_vs_accuracy_{block_type}.png",
                 dpi = 130)
    plt.show()


## 11.  Section G -- sub-sample power diagnostics

For each "trending" comparison we ask: at the observed effect size,
would more subjects have made this significant, or is the effect
itself too small to be reliable?  We sub-sample: at each candidate
sub-sample size $n$, draw 2000 random subsets of that size from the
observed data, run the relevant test, and record the fraction with
$p < 0.05$.  This estimates the power that the *observed* effect would
deliver at sample size $n$.

- **G1.**  Oberlin full_sentence accuracy vs chance (one-sample
  Wilcoxon; expected to be easily powered).
- **G2.**  Site difference, full_sentence (unpaired Mann-Whitney;
  sub-sample preserves Oberlin/UChicago ratio).
- **G3.**  Full vs Imagined paired (paired Wilcoxon on within-subject
  differences).

In [ ]:
# ============================================================================
# Section G -- sub-sampling power for a one-sample Wilcoxon test
# ============================================================================

def subsample_power_one_sample(values, null_mean = 0.5,
                                sub_sample_sizes = None,
                                n_resample = 2000, alpha = 0.05,
                                rng = RNG):
    if sub_sample_sizes is None:
        sub_sample_sizes = list(range(5, len(values) + 1))
    rows = []
    for n_sub in sub_sample_sizes:
        n_significant    = 0
        sub_sample_means = []
        for _ in range(n_resample):
            sub_sample = rng.choice(values, size = n_sub, replace = False)
            try:
                _, p_value = wilcoxon(sub_sample - null_mean)
            except ValueError:
                continue
            sub_sample_means.append(np.mean(sub_sample))
            if p_value < alpha:
                n_significant += 1
        rows.append(dict(n_sub         = n_sub,
                          power         = n_significant / n_resample,
                          mean_of_means = float(np.mean(sub_sample_means)),
                          sd_of_means   = float(
                              np.std(sub_sample_means, ddof = 1))))
    return pd.DataFrame(rows)


In [ ]:
# ============================================================================
# Section G -- sub-sampling power for an unpaired Mann-Whitney test
#   Sub-sample preserves the Oberlin / UChicago ratio (~16/26).
# ============================================================================

def subsample_power_two_sample(group_a, group_b, n_resample = 2000,
                                alpha = 0.05, rng = RNG):
    base_ratio = len(group_a) / (len(group_a) + len(group_b))
    rows = []
    for total_n in range(10, len(group_a) + len(group_b) + 1):
        n_a = max(3, int(round(total_n * base_ratio)))
        n_b = max(3, total_n - n_a)
        if n_a > len(group_a) or n_b > len(group_b):
            continue
        n_significant    = 0
        sub_sample_diffs = []
        for _ in range(n_resample):
            sub_a = rng.choice(group_a, size = n_a, replace = False)
            sub_b = rng.choice(group_b, size = n_b, replace = False)
            mwu_stat, p_value = mannwhitneyu(sub_a, sub_b,
                                              alternative = "two-sided")
            sub_sample_diffs.append(np.mean(sub_a) - np.mean(sub_b))
            if p_value < alpha:
                n_significant += 1
        rows.append(dict(n_total   = total_n,
                          n_a       = n_a,
                          n_b       = n_b,
                          power     = n_significant / n_resample,
                          mean_diff = float(np.mean(sub_sample_diffs)),
                          sd_diff   = float(
                              np.std(sub_sample_diffs, ddof = 1))))
    return pd.DataFrame(rows)


In [ ]:
# ============================================================================
# Section G -- sub-sampling power for a paired Wilcoxon test
#   Operates on the per-subject within-subject differences.
# ============================================================================

def subsample_power_paired(differences, sub_sample_sizes = None,
                            n_resample = 2000, alpha = 0.05, rng = RNG):
    if sub_sample_sizes is None:
        sub_sample_sizes = list(range(5, len(differences) + 1))
    rows = []
    for n_sub in sub_sample_sizes:
        n_significant    = 0
        sub_sample_means = []
        for _ in range(n_resample):
            sub_sample = rng.choice(differences, size = n_sub,
                                     replace = False)
            try:
                _, p_value = wilcoxon(sub_sample)
            except ValueError:
                continue
            sub_sample_means.append(np.mean(sub_sample))
            if p_value < alpha:
                n_significant += 1
        rows.append(dict(n_sub     = n_sub,
                          power     = n_significant / n_resample,
                          mean_diff = float(np.mean(sub_sample_means))))
    return pd.DataFrame(rows)


In [ ]:
# ============================================================================
# Section G -- run the three sub-sample power analyses
# ============================================================================

print("\n=== Section G: N-sensitivity / subsample power curves ===")

# G1: Oberlin full_sentence accuracy vs chance
oberlin_full_sentence_accuracies = per_subject_table.query(
    "site == 'Oberlin' and block_type == 'full_sentence'")["accuracy"].values
g1_power_table = subsample_power_one_sample(
    oberlin_full_sentence_accuracies, null_mean = 0.5)
g1_power_table.to_csv(TABLES_PATH / "G_power_oberlin_fs_vs_chance.csv",
                       index = False)

# G2: site difference (Oberlin vs UChicago), full_sentence
accuracy_oberlin_full  = per_subject_table.query(
    "site == 'Oberlin'  and block_type == 'full_sentence'")["accuracy"].values
accuracy_uchicago_full = per_subject_table.query(
    "site == 'UChicago' and block_type == 'full_sentence'")["accuracy"].values
g2_power_table = subsample_power_two_sample(accuracy_oberlin_full,
                                              accuracy_uchicago_full)
g2_power_table.to_csv(TABLES_PATH / "G_power_site_full_sentence.csv",
                       index = False)

# G3: Full vs Imagined paired (Unified pool)
wide_unified = per_subject_table.pivot(index   = "subject",
                                        columns = "block_type",
                                        values  = "accuracy").dropna()
unified_full_minus_imagined = (wide_unified["full_sentence"]
                                - wide_unified["imagined_sentence"]).values
g3_power_table = subsample_power_paired(unified_full_minus_imagined)
g3_power_table.to_csv(TABLES_PATH / "G_power_full_vs_imagined_unified.csv",
                       index = False)

print("g1 head:\n", g1_power_table.head().round(3).to_string(index = False))
print("g2 head:\n", g2_power_table.head().round(3).to_string(index = False))
print("g3 head:\n", g3_power_table.head().round(3).to_string(index = False))


In [ ]:
# ============================================================================
# Section G -- combined power-curve figure (3 panels side by side)
# ============================================================================

fig, axes_row = plt.subplots(1, 3, figsize = (13, 4.2))

ax = axes_row[0]
ax.plot(g1_power_table["n_sub"], g1_power_table["power"], marker = "o",
         color = SITE_COLOR["Oberlin"])
ax.axhline(0.80, color = "red",  linestyle = "--", alpha = 0.7,
            label = "0.80 power")
ax.axhline(0.05, color = "gray", linestyle = ":",  alpha = 0.7,
            label = "alpha = 0.05")
ax.set_xlabel("subsample size n")
ax.set_ylabel("Pr(Wilcoxon p < 0.05)")
ax.set_title("G1.  Oberlin full_sentence accuracy vs chance\n"
              f"(subsampling from observed N = "
              f"{len(oberlin_full_sentence_accuracies)})", fontsize = 10)
ax.legend(loc = "lower right", fontsize = 8)
ax.set_ylim(0, 1)

ax = axes_row[1]
ax.plot(g2_power_table["n_total"], g2_power_table["power"], marker = "o",
         color = "purple")
ax.axhline(0.80, color = "red",  linestyle = "--", alpha = 0.7)
ax.axhline(0.05, color = "gray", linestyle = ":",  alpha = 0.7)
ax.set_xlabel("total subsample size (Oberlin + UChicago)")
ax.set_ylabel("Pr(MWU p < 0.05)")
ax.set_title("G2.  Site difference, full_sentence accuracy\n"
              f"(observed nA = {len(accuracy_oberlin_full)}, "
              f"nB = {len(accuracy_uchicago_full)})", fontsize = 10)
ax.set_ylim(0, 1)

ax = axes_row[2]
ax.plot(g3_power_table["n_sub"], g3_power_table["power"], marker = "o",
         color = SITE_COLOR["Unified"])
ax.axhline(0.80, color = "red",  linestyle = "--", alpha = 0.7)
ax.axhline(0.05, color = "gray", linestyle = ":",  alpha = 0.7)
ax.set_xlabel("paired subsample size n")
ax.set_ylabel("Pr(paired Wilcoxon p < 0.05)")
ax.set_title("G3.  Full vs Imagined within-subject\n"
              f"(observed N = {len(unified_full_minus_imagined)})",
              fontsize = 10)
ax.set_ylim(0, 1)

plt.suptitle("Section G.  Sub-sample 'is it a power problem?' diagnostics",
              fontsize = 12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "G_power_curves.png", dpi = 140)
plt.show()


In [ ]:
# ============================================================================
# Section G -- empirical "N for 80% power" summary table
#   Linear-scan: smallest sub-sample size at which empirical power >= 0.80.
#   Returns None (-> empty CSV cell) if the curve never crosses 0.80.
# ============================================================================

def n_for_target_power(power_table, n_column = "n_sub", target_power = 0.80):
    above_target = power_table[power_table["power"] >= target_power]
    if len(above_target) == 0:
        return None
    return int(above_target.iloc[0][n_column])


required_n_rows = [
    dict(comparison        = "Oberlin full_sentence vs chance",
         observed_n        = len(oberlin_full_sentence_accuracies),
         observed_p        = float(accuracy_vs_chance_table.query(
             "block == 'full_sentence' and group == 'Oberlin'"
         ).iloc[0]["wilcoxon_p"]),
         observed_d        = float(accuracy_vs_chance_table.query(
             "block == 'full_sentence' and group == 'Oberlin'"
         ).iloc[0]["cohens_d"]),
         n_for_80pct_power = n_for_target_power(g1_power_table, "n_sub")),
    dict(comparison        = "Site Oberlin vs UChicago, full_sentence",
         observed_n        = (len(accuracy_oberlin_full)
                               + len(accuracy_uchicago_full)),
         observed_p        = float(site_comparison_table.query(
             "block == 'full_sentence'").iloc[0]["mwu_p"]),
         observed_d        = float(site_comparison_table.query(
             "block == 'full_sentence'").iloc[0]["cohens_d"]),
         n_for_80pct_power = n_for_target_power(g2_power_table, "n_total")),
    dict(comparison        = "Full vs Imagined paired (Unified)",
         observed_n        = len(unified_full_minus_imagined),
         observed_p        = float(full_vs_imagined_table.query(
             "group == 'Unified'").iloc[0]["wilcoxon_p"]),
         observed_d        = float(full_vs_imagined_table.query(
             "group == 'Unified'").iloc[0]["cohens_d_paired"]),
         n_for_80pct_power = n_for_target_power(g3_power_table, "n_sub"))]

required_n_table = pd.DataFrame(required_n_rows)
required_n_table.to_csv(TABLES_PATH / "G_required_n_summary.csv",
                         index = False)

print("\n--- N required for 80% power "
      "(interpolated from subsample curves) ---")
print(required_n_table.to_string(index = False))


## 12.  Done

All CSVs and PNGs that the report consumes are now refreshed:

- `tables/A_accuracy_vs_chance.csv`          + `figures/A_accuracy_vs_chance.png`
- `tables/B_full_vs_imagined.csv`            + `figures/B_full_vs_imagined.png`
- `tables/C_site_comparison.csv`             + `figures/C_site_comparison.png`
- `tables/D_block_order.csv`
- `tables/E_convergent_full_sentence.csv`    + `figures/E_convergent_full_sentence.png`
- `tables/F_questionnaire_spearman.csv`      + `figures/F_quest_vs_accuracy_{block}.png`
- `tables/G_power_oberlin_fs_vs_chance.csv`
- `tables/G_power_site_full_sentence.csv`
- `tables/G_power_full_vs_imagined_unified.csv`
- `tables/G_required_n_summary.csv`          + `figures/G_power_curves.png`

In [ ]:
print(f"\nDone.  Tables -> {TABLES_PATH}")
print(f"        Figures -> {FIGURES_PATH}")
